# Laundrmate Advertisement — FFmpeg Merge Notebook (Google Colab)

Merges the numbered Higgsfield clips (`clip_01_*.mp4` … `clip_14_*.mp4`) into one
seamless 60-second master, then muxes the three language voiceover tracks to produce:

- `Laundrmate_Advertisement_Tamil.mp4`
- `Laundrmate_Advertisement_English.mp4`
- `Laundrmate_Advertisement_Hindi.mp4`

**How it avoids the classic merge problems:**

- *Frame-rate changes* → every clip is first re-encoded to a constant frame rate
  (CFR), identical resolution, codec, and pixel format before concatenation.
- *Freezing / black gaps* → uniform re-encoded streams are joined with the concat
  demuxer (no stream-copy of mismatched streams); a QC cell runs `blackdetect` and
  `freezedetect` on the output.
- *Audio sync problems* → the master is built silent; each language VO + music bed
  is mixed and muxed afterwards with `aresample=async=1` and `-shortest`.

**Inputs expected** (upload or place in Google Drive):

```
clips/clip_01_opening_banner.mp4 … clips/clip_14_final_cta.mp4   (exact numerical order)
audio/vo_tamil.wav  audio/vo_english.wav  audio/vo_hindi.wav
audio/music_bed.wav   (optional — shared music/SFX bed)
```

In [ ]:
# 1) Setup — ffmpeg is preinstalled on Colab; this just confirms the version.
!ffmpeg -version | head -n 1

import os, glob, re, json, subprocess

# Option A: mount Google Drive (uncomment if clips live in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/Laundrmate'

# Option B: work locally in the Colab session (upload via the Files panel)
BASE = '/content/Laundrmate'

CLIPS_DIR  = os.path.join(BASE, 'clips')
AUDIO_DIR  = os.path.join(BASE, 'audio')
WORK_DIR   = os.path.join(BASE, 'normalized')
OUT_DIR    = os.path.join(BASE, 'output')
for d in (CLIPS_DIR, AUDIO_DIR, WORK_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# Target master format — keep constant everywhere.
FPS        = 25          # constant frame rate for the whole ad
WIDTH, HEIGHT = 1920, 1080
EXPECTED_CLIPS = 14
print('Working under:', BASE)

In [ ]:
# 2) Verify all clips are present, in exact numerical order, and readable.
clips = sorted(glob.glob(os.path.join(CLIPS_DIR, 'clip_*.mp4')))
assert clips, f'No clips found in {CLIPS_DIR} — upload clip_01…clip_14 first.'

nums = [int(re.search(r'clip_(\\d+)', os.path.basename(c)).group(1)) for c in clips]
missing = sorted(set(range(1, EXPECTED_CLIPS + 1)) - set(nums))
assert not missing, f'Missing clip numbers: {missing}'
assert nums == sorted(nums) and len(nums) == len(set(nums)), 'Clip numbering must be unique and ordered.'

def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                        '-show_entries', 'stream=width,height,r_frame_rate,codec_name',
                        '-show_entries', 'format=duration', '-of', 'json', path],
                       capture_output=True, text=True)
    return json.loads(r.stdout)

total = 0.0
for c in clips:
    info = probe(c)
    dur = float(info['format']['duration'])
    st = info['streams'][0]
    total += dur
    print(f"{os.path.basename(c):40s} {st['width']}x{st['height']} {st['r_frame_rate']:>8s} {st['codec_name']:5s} {dur:6.2f}s")
print(f'\\nTotal raw duration: {total:.2f}s (target ≈ 60s)')

In [ ]:
# 3) Normalize every clip to identical CFR / resolution / codec / pixel format.
#    This is what prevents freezes, stutters and frame-rate jumps at the joins.
normalized = []
for c in clips:
    out = os.path.join(WORK_DIR, os.path.basename(c))
    cmd = ['ffmpeg', '-y', '-i', c,
           '-vf', (f'scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease,'
                   f'pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,'
                   f'fps={FPS},format=yuv420p'),
           '-r', str(FPS), '-fps_mode', 'cfr',
           '-c:v', 'libx264', '-preset', 'slow', '-crf', '17',
           '-an',                     # master video is built silent; audio muxed later
           '-movflags', '+faststart', out]
    print('Normalizing', os.path.basename(c))
    subprocess.run(cmd, check=True, capture_output=True)
    normalized.append(out)
print('Normalized', len(normalized), 'clips.')

In [ ]:
# 4) Concatenate into the silent master (streams are now uniform, so the
#    concat demuxer joins them without gaps, freezes or timestamp jumps).
concat_list = os.path.join(WORK_DIR, 'concat.txt')
with open(concat_list, 'w') as f:
    for n in normalized:
        f.write(f"file '{n}'\\n")

MASTER = os.path.join(OUT_DIR, 'Laundrmate_Master_Silent.mp4')
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list,
                '-c:v', 'libx264', '-preset', 'slow', '-crf', '17',
                '-r', str(FPS), '-fps_mode', 'cfr', '-pix_fmt', 'yuv420p',
                '-movflags', '+faststart', MASTER], check=True, capture_output=True)
print('Master written:', MASTER)
print(probe(MASTER)['format']['duration'], 'seconds')

In [ ]:
# 5) Mux the three language versions (same visuals; only VO/subtitles change).
#    aresample=async=1 gently corrects small audio drift → no sync problems.
LANGS = {
    'Tamil':   os.path.join(AUDIO_DIR, 'vo_tamil.wav'),
    'English': os.path.join(AUDIO_DIR, 'vo_english.wav'),
    'Hindi':   os.path.join(AUDIO_DIR, 'vo_hindi.wav'),
}
MUSIC = os.path.join(AUDIO_DIR, 'music_bed.wav')  # optional
have_music = os.path.exists(MUSIC)

for lang, vo in LANGS.items():
    if not os.path.exists(vo):
        print(f'!! Skipping {lang}: {vo} not found')
        continue
    out = os.path.join(OUT_DIR, f'Laundrmate_Advertisement_{lang}.mp4')
    if have_music:
        # duck music under the voiceover, then resample-correct sync
        cmd = ['ffmpeg', '-y', '-i', MASTER, '-i', vo, '-i', MUSIC,
               '-filter_complex',
               '[2:a]volume=0.25[m];[1:a][m]amix=inputs=2:duration=first:dropout_transition=0,'
               'aresample=async=1:first_pts=0[a]',
               '-map', '0:v:0', '-map', '[a]']
    else:
        cmd = ['ffmpeg', '-y', '-i', MASTER, '-i', vo,
               '-filter_complex', '[1:a]aresample=async=1:first_pts=0[a]',
               '-map', '0:v:0', '-map', '[a]']
    cmd += ['-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', '-shortest',
            '-movflags', '+faststart', out]
    subprocess.run(cmd, check=True, capture_output=True)
    print('Wrote', out)

In [ ]:
# 6) QC — fail loudly on black gaps, freezes, or duration/fps problems.
import shlex

def qc(path):
    print('===', os.path.basename(path))
    info = probe(path)
    dur = float(info['format']['duration'])
    st = info['streams'][0]
    print(f"  duration {dur:.2f}s | {st['width']}x{st['height']} | fps {st['r_frame_rate']} | {st['codec_name']}")
    assert 55 <= dur <= 65, f'Duration {dur:.2f}s is outside the ~60s target!'
    for name, flt in (('black frames', 'blackdetect=d=0.1:pix_th=0.10'),
                      ('frozen video', 'freezedetect=n=-60dB:d=0.5')):
        r = subprocess.run(f'ffmpeg -i {shlex.quote(path)} -vf {flt} -an -f null - 2>&1',
                           shell=True, capture_output=True, text=True)
        hits = [l for l in r.stdout.splitlines() if 'black_start' in l or 'freeze_start' in l]
        print(f'  {name}: {"NONE ✅" if not hits else "DETECTED ❌"}')
        for h in hits:
            print('   ', h.strip())

for f in sorted(glob.glob(os.path.join(OUT_DIR, 'Laundrmate_Advertisement_*.mp4'))):
    qc(f)
print('\\nQC complete. Also verify manually against production/continuity-checklist.md:')
print(' - banner at start, phone 98847 12121, offer dates 31-07-2026 to 07-08-2026,')
print(' - consistent characters, realistic machines, no AI artefacts, no spelling mistakes.')

## Notes

- If any clip is re-generated in Higgsfield, re-upload it with the **same numbered
  filename** and re-run from cell 3 — the pipeline is deterministic.
- If clips arrive slightly long/short, trim at generation time or add `-ss`/`-t` to
  the normalize command for that clip; keep scene boundaries on the timing table in
  `production/shot-list.md`.
- Burned-in subtitles (optional): add
  `-vf subtitles=subs_<lang>.srt` while muxing each language version (re-encode
  video with `-c:v libx264 -crf 17` instead of `-c:v copy` in that case).